# Renk Uzayları ve Renk Tabanlı Görüntü Segmentasyonu

Bu modül; bilgisayarlı görüde nesne ayrıştırmanın en temel yöntemlerinden biri olan **renk uzayları (BGR, RGB, HSV, LAB)** arasındaki dönüşümleri ve HSV uzayında renk maskeleme ile hedef nesnelerin arka plandan izole edilmesini inceler.

---

## 1. Neden BGR Yerine HSV?

- **BGR / RGB:** Kırmızı, Yeşil ve Mavi kanalları donanım düzeyinde temsil eder. Ancak ışık şiddeti (parlaklık) değiştiğinde $R, G, B$ değerlerinin üçü birden değişir. Bu durum gölgeli ve parlamalı ortamlarda renk filtrelemeyi imkansız kılar.
- **HSV Uzayı:**
  - **H (Hue - Renk Özü):** Rengin dalga boyunu belirler ($0^\circ - 360^\circ$, OpenCV'de 1 bayta sığması için $[0, 179]$ aralığına ölçeklenir).
  - **S (Saturation - Doygunluk):** Rengin saflık/canlılık oranı ($[0, 255]$).
  - **V (Value - Parlaklık):** Işık yoğunluğu ($[0, 255]$).
  
HSV uzayında parlaklık ($V$) renkten ($H$) tamamen bağımsız bir boyuttur. Bu sayede aydınlatma dalgalanmalarından etkilenmeden stabil renk tespiti yapılabilir.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Test görselini yükleme (BGR)
image = cv2.imread('color_spheres.png')

# Renk Uzayı Dönüşümleri
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image_hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
image_lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_rgb)
axes[0].set_title("RGB Renk Uzayı")
axes[0].axis('off')

axes[1].imshow(image_hsv)
axes[1].set_title("HSV Temsili (H, S, V)")
axes[1].axis('off')

axes[2].imshow(image_lab)
axes[2].set_title("CIE-LAB Temsili (L, A, B)")
axes[2].axis('off')
plt.show()


## 2. HSV Kanallarının Ayrı Ayrı İncelenmesi

In [ ]:
h, s, v = cv2.split(image_hsv)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(h, cmap='hsv')
axes[0].set_title("Hue (Renk Özü: 0-179)")
axes[0].axis('off')

axes[1].imshow(s, cmap='gray')
axes[1].set_title("Saturation (Doygunluk: 0-255)")
axes[1].axis('off')

axes[2].imshow(v, cmap='gray')
axes[2].set_title("Value (Parlaklık: 0-255)")
axes[2].axis('off')
plt.show()


## 3. Renk Maskesi ile Hedef Nesneyi İzole Etme

Örnek olarak görüntüdeki **Mavi Nesneyi** izole edelim:
OpenCV HSV renk skalasında Mavi yaklaşık $H \in [100, 130]$ aralığındadır.


In [ ]:
# Mavi renk sınırları (H, S, V)
lower_blue = np.array([100, 100, 100])
upper_blue = np.array([130, 255, 255])

# İkili Maske Çıkarma
blue_mask = cv2.inRange(image_hsv, lower_blue, upper_blue)

# Bitwise AND işlemi ile sadece mavi nesneyi alma
blue_segmented = cv2.bitwise_and(image_rgb, image_rgb, mask=blue_mask)

# Kırmızı Renk Döngüsü (Red Wrap-around)
# Kırmızı hem 0-10 hem 170-180 aralığında yer alır
lower_red1 = np.array([0, 120, 70])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 120, 70])
upper_red2 = np.array([180, 255, 255])

mask_red1 = cv2.inRange(image_hsv, lower_red1, upper_red1)
mask_red2 = cv2.inRange(image_hsv, lower_red2, upper_red2)
red_mask = cv2.bitwise_or(mask_red1, mask_red2)
red_segmented = cv2.bitwise_and(image_rgb, image_rgb, mask=red_mask)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].imshow(blue_mask, cmap='gray')
axes[0, 0].set_title("Mavi İkili Maske (cv2.inRange)")
axes[0, 0].axis('off')

axes[0, 1].imshow(blue_segmented)
axes[0, 1].set_title("İzole Edilmiş Mavi Nesne")
axes[0, 1].axis('off')

axes[1, 0].imshow(red_mask, cmap='gray')
axes[1, 0].set_title("Kırmızı Çift Aralık Maskesi")
axes[1, 0].axis('off')

axes[1, 1].imshow(red_segmented)
axes[1, 1].set_title("İzole Edilmiş Kırmızı Nesne")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()
